[Reference](https://leapcell.medium.com/how-to-build-a-search-engine-from-scratch-in-python-no-external-packages-36cc9e0ba9d0)

# Step 1: Document Preprocessing

In [1]:
import string

# Define English stop words set
STOP_WORDS = {
    'a', 'an', 'and', 'the', 'or', 'of', 'to', 'in', 'for', 'on', 'with',
    'at', 'by', 'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him',
    'her', 'us', 'them', 'my', 'your', 'his', 'its', 'our', 'their', 'this',
    'that', 'these', 'those', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'shall', 'should',
    'may', 'might', 'must', 'can', 'could', 'as', 'but', 'if', 'or', 'because',
    'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'from', 'up',
    'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then',
    'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
    'only', 'own', 'same', 'so', 'than', 'too', 'very'
}
def preprocess_text(text):
    """Preprocess text: case conversion, punctuation removal, tokenization, stop word removal"""
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    # Tokenization (simple space splitting; more complex logic can be used in practical applications)
    tokens = text.split()
    # Remove stop words and empty strings
    tokens = [token for token in tokens if token not in STOP_WORDS and token.strip() != '']
    # Simple stemming (simplified version)
    tokens = [stem_token(token) for token in tokens]
    return tokens
def stem_token(token):
    """Simple stemming function (more complex algorithms can be used in practical applications)"""
    # Handle common suffixes
    suffixes = ['ing', 'ly', 'ed', 'es', 's']
    for suffix in suffixes:
        if token.endswith(suffix) and len(token) > len(suffix):
            return token[:-len(suffix)]
    return token
# Test the preprocessing function
sample_text = "Machine learning is a subset of artificial intelligence focused on developing algorithms that learn from data."
processed_tokens = preprocess_text(sample_text)
print("Preprocessed terms:", processed_tokens)

Preprocessed terms: ['machine', 'learn', 'subset', 'artificial', 'intelligence', 'focus', 'develop', 'algorithm', 'learn', 'data']


# Step 2: Build Inverted Index and Store as CSV

In [2]:
import csv
from collections import defaultdict

def build_inverted_index(documents):
    """Build inverted index and store as CSV file"""
    inverted_index = defaultdict(list)  # Structure: {term: [(doc_id, positions), ...]}
    for doc_id, doc in enumerate(documents):
        # Preprocess the document
        tokens = preprocess_text(doc)
        # Record positions of each term in the current document
        term_positions = defaultdict(list)
        for pos, term in enumerate(tokens):
            term_positions[term].append(pos)
        # Update inverted index
        for term, positions in term_positions.items():
            inverted_index[term].append((doc_id, positions))
    # Store inverted index as CSV file
    with open('inverted_index.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['term', 'doc_id', 'positions'])
        for term, doc_info in inverted_index.items():
            for doc_id, positions in doc_info:
                # Convert position list to string for storage
                positions_str = str(positions)
                writer.writerow([term, doc_id, positions_str])
    return inverted_index
# Sample document collection
documents = [
    "Machine learning is a subset of artificial intelligence focused on developing algorithms that learn from data.",
    "Artificial intelligence involves creating systems that can perform tasks requiring human intelligence.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers.",
    "Natural language processing allows computers to understand and generate human language.",
    "Computer vision enables machines to interpret and understand the visual world.",
    "Reinforcement learning is an area of machine learning concerned with how agents take actions in an environment.",
    "Supervised learning algorithms learn from labeled training data to make predictions on new data.",
    "Unsupervised learning deals with unlabeled data, finding patterns and structures within it.",
    "A neural network is a computational model inspired by the human brain's structure and function.",
    "Big data refers to large and complex data sets that require advanced processing techniques."
]
# Build inverted index
inverted_index = build_inverted_index(documents)
print(f"Inverted index built, containing {len(inverted_index)} terms")

Inverted index built, containing 70 terms


# Step 3: Calculate TF-IDF Values

In [3]:
def calculate_tfidf(documents, inverted_index):
    """Calculate TF-IDF values for each term in each document"""
    num_docs = len(documents)
    tfidf = {}  # Structure: {doc_id: {term: tfidf_value, ...}, ...}

    # Calculate total number of terms for each document
    doc_lengths = []
    for doc in documents:
        tokens = preprocess_text(doc)
        doc_lengths.append(len(tokens))
    # Calculate document frequency for each term (number of documents containing the term)
    doc_freq = {term: len(entries) for term, entries in inverted_index.items()}
    # Calculate TF-IDF
    for term, entries in inverted_index.items():
        # Calculate IDF
        idf = math.log(num_docs / (doc_freq[term] + 1))  # +1 to avoid division by zero
        for doc_id, positions in entries:
            # Calculate TF
            tf = len(positions) / doc_lengths[doc_id] if doc_lengths[doc_id] > 0 else 0
            # Calculate TF-IDF
            tfidf_value = tf * idf
            # Store results
            if doc_id not in tfidf:
                tfidf[doc_id] = {}
            tfidf[doc_id][term] = tfidf_value
    return tfidf
import math  # Import math library for logarithm calculation
# Calculate TF-IDF values
tfidf_scores = calculate_tfidf(documents, inverted_index)
print("TF-IDF calculation completed")

TF-IDF calculation completed


# Step 4: Process Queries and Return Results

In [4]:
def search(query, documents, inverted_index, tfidf_scores, top_n=3):
    """Process query and return most relevant documents"""
    # Preprocess query
    query_terms = preprocess_text(query)
    if not query_terms:
        return []

    # Get documents containing at least one query term
    relevant_docs = set()
    for term in query_terms:
        if term in inverted_index:
            for doc_id, _ in inverted_index[term]:
                relevant_docs.add(doc_id)
    relevant_docs = list(relevant_docs)
    # Calculate relevance scores between query and each relevant document
    scores = []
    for doc_id in relevant_docs:
        score = 0.0
        for term in query_terms:
            if term in tfidf_scores.get(doc_id, {}):
                score += tfidf_scores[doc_id][term]
        # Normalize score (divide by number of query terms)
        score /= len(query_terms)
        scores.append((doc_id, score))
    # Sort by score
    scores.sort(key=lambda x: x[1], reverse=True)
    # Return top N results
    results = []
    for doc_id, score in scores[:top_n]:
        if score > 0:
            results.append({
                'document': documents[doc_id],
                'score': score,
                'doc_id': doc_id
            })
    return results
# Test search functionality
import math  # Ensure math library is imported
query = "machine learning"
results = search(query, documents, inverted_index, tfidf_scores)
print(f"Query: {query}")
for i, result in enumerate(results, 1):
    print(f"\n Result {i} (Score: {result['score']:.4f}):")
    print(result['document'])

Query: machine learning

 Result 1 (Score: 0.0969):
Machine learning is a subset of artificial intelligence focused on developing algorithms that learn from data.

 Result 2 (Score: 0.0969):
Reinforcement learning is an area of machine learning concerned with how agents take actions in an environment.

 Result 3 (Score: 0.0881):
Deep learning is a type of machine learning based on artificial neural networks with multiple layers.


# Step 5: Load Inverted Index from CSV

In [5]:
def load_inverted_index_from_csv(filename):
    """Load inverted index from CSV file"""
    inverted_index = defaultdict(list)

    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            term = row[0]
            doc_id = int(row[1])
            # Convert position string back to list
            positions = eval(row[2])  # Note: eval has security risks; use safer methods in practical applications
            inverted_index[term].append((doc_id, positions))
    return inverted_index
# Test loading inverted index
loaded_index = load_inverted_index_from_csv('inverted_index.csv')
print(f"Inverted index loaded from CSV contains {len(loaded_index)} terms")

Inverted index loaded from CSV contains 70 terms
